# Exercise: 2D Scattered-Data Interpolation with Radial Basis Functions

**Instructions:** Fill in every cell marked `# TODO`. Each TODO cell is followed by a *sanity check* cell (do not modify those) — run it right after your implementation to get instant feedback. If a check fails, you'll get a clear `AssertionError`; if it passes, you'll see an `OK ✅` message.

**Learning objectives**
- Understand RBF interpolation as solving a linear system $\Phi \mathbf{w} = \mathbf{f}$.
- Practice vectorized (loop-free) pairwise-distance computations with NumPy broadcasting.
- See how the interpolant is just a weighted sum of "bumps" centered at the data points.
- Explore how the kernel width (`epsilon`) trades off smoothness against accuracy.

**Background.** Given scattered points $\mathbf{x}_i \in \mathbb{R}^2$ with known values $f_i = f(\mathbf{x}_i)$, we build an interpolant

$$s(\mathbf{x}) = \sum_{i=1}^N w_i\, \varphi(\lVert \mathbf{x} - \mathbf{x}_i \rVert)$$

where $\varphi$ is a radial basis function (here a Gaussian, $\varphi(r) = e^{-(\epsilon r)^2}$). The weights $w_i$ are found by requiring $s(\mathbf{x}_j) = f_j$ for every data point, i.e. solving

$$\Phi \mathbf{w} = \mathbf{f}, \qquad \Phi_{ij} = \varphi(\lVert \mathbf{x}_i - \mathbf{x}_j \rVert)$$

No mesh is needed — just the scattered points.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Scattered sample points and a test function

We pick a smooth function $f(x, y)$ and sample it at $N$ random scattered points — pretend that's all we know. *(Given — nothing to fill in here.)*

In [ ]:
def f(x, y):
    return np.sin(3 * x) * np.cos(3 * y) + 0.5 * x

rng = np.random.default_rng(42)
N = 40
pts = rng.uniform(-1, 1, size=(N, 2))   # scattered (x, y) points
vals = f(pts[:, 0], pts[:, 1])           # known values at those points

## 2. Pairwise distances

**TODO:** implement `pairwise_dist(A, B)`, which returns an $(M, N)$ array `D` where `D[i, j]` is the Euclidean distance between `A[i]` and `B[j]`, for `A` of shape `(M, 2)` and `B` of shape `(N, 2)`.

Do this with NumPy broadcasting — **no explicit Python `for` loops**.

*Hint:* `A[:, None, :] - B[None, :, :]` has shape `(M, N, 2)` — the last axis holds the $(x, y)$ difference for every pair. Square it, sum over the last axis, take the square root.

In [ ]:
def pairwise_dist(A, B):
    """Euclidean distance between every point in A and every point in B.

    A: array of shape (M, 2)
    B: array of shape (N, 2)
    returns: array of shape (M, N) where result[i, j] = ||A[i] - B[j]||
    """
    # TODO: implement using broadcasting
    raise NotImplementedError("pairwise_dist is not implemented yet")

In [ ]:
# Sanity check — do not modify
test_A = np.array([[0.0, 0.0], [1.0, 0.0]])
test_B = np.array([[0.0, 0.0], [0.0, 1.0]])
D = pairwise_dist(test_A, test_B)
expected = np.array([[0.0, 1.0],
                      [1.0, np.sqrt(2.0)]])

assert D.shape == (2, 2), f"expected shape (2, 2), got {D.shape}"
assert np.allclose(D, expected), f"pairwise_dist looks wrong:\ngot:\n{D}\nexpected:\n{expected}"
print("pairwise_dist OK \u2705")

## 3. The Gaussian radial basis function

**TODO:** implement `gaussian_rbf(r, epsilon)`, computing $\varphi(r) = e^{-(\epsilon r)^2}$ elementwise. `r` can be a scalar or a NumPy array — your implementation should work for both (that's automatic if you just use NumPy elementwise ops).

In [ ]:
def gaussian_rbf(r, epsilon=3.0):
    # TODO: implement phi(r) = exp(-(epsilon * r)**2)
    raise NotImplementedError("gaussian_rbf is not implemented yet")

In [ ]:
# Sanity check — do not modify
r_test = np.array([0.0, 10.0])
phi_test = gaussian_rbf(r_test, epsilon=3.0)

assert np.isclose(phi_test[0], 1.0), "phi(0) should be exactly 1"
assert phi_test[1] < 1e-10, "phi(r) should decay to ~0 for large r"
print("gaussian_rbf OK \u2705")

## 4. Solve for the interpolation weights

**TODO:**
1. Build the $N \times N$ matrix `Phi` with `Phi[i, j] = phi(||x_i - x_j||)`, using the two functions you just wrote.
2. Solve the linear system $\Phi \mathbf{w} = \mathbf{f}$ for `weights` (look at `np.linalg.solve`).

In [ ]:
epsilon = 3.0

# TODO: build Phi (shape (N, N)) using pairwise_dist and gaussian_rbf on `pts`
Phi = ...

# TODO: solve Phi @ weights = vals for weights
weights = ...

In [ ]:
# Sanity check — do not modify
assert Phi.shape == (N, N), f"Phi should be ({N}, {N}), got {Phi.shape}"
assert np.allclose(Phi, Phi.T), "Phi should be symmetric (distance is symmetric)"
assert np.allclose(np.diag(Phi), 1.0), "Phi's diagonal should be phi(0) = 1"

recovered = Phi @ weights
assert np.allclose(recovered, vals, atol=1e-6), (
    "The interpolant should reproduce the training data exactly at the data points "
    "(max residual = %.2e). Check your linear solve." % np.max(np.abs(recovered - vals))
)
print("weights OK \u2705  (max residual at data points: %.2e)" % np.max(np.abs(recovered - vals)))

## 5. Evaluate the interpolant on a fine grid

**TODO:** evaluate $s(\mathbf{x}) = \Phi(\mathbf{x}, X)\, \mathbf{w}$ on the grid of query points `grid_pts` (already built for you below):
1. Compute the distance from every grid point to every data point.
2. Turn those distances into RBF values.
3. Multiply by `weights` and reshape the result back onto the grid (shape `Xg.shape`).

In [ ]:
nx, ny = 150, 150
xg = np.linspace(-1, 1, nx)
yg = np.linspace(-1, 1, ny)
Xg, Yg = np.meshgrid(xg, yg)
grid_pts = np.column_stack([Xg.ravel(), Yg.ravel()])

# TODO: distances from grid_pts to pts
Rg = ...

# TODO: RBF values for those distances
Phig = ...

# TODO: interpolated values, reshaped to match Xg's shape
Zg = ...

Ztrue = f(Xg, Yg)  # ground truth, for comparison only

In [ ]:
# Sanity check — do not modify
assert Rg.shape == (nx * ny, N), f"Rg should be ({nx*ny}, {N}), got {Rg.shape}"
assert Phig.shape == Rg.shape
assert Zg.shape == Xg.shape, f"Zg should be reshaped to {Xg.shape}, got {Zg.shape}"

err = np.abs(Zg - Ztrue)
print("grid evaluation OK \u2705  (mean abs error vs. true function: %.4f)" % err.mean())

## 6. Visualize as 3D surfaces

**TODO:** the middle plot (`ax1`) should show the RBF interpolant *and* the original data points at their true heights, so you can see the surface passing through them. Add a `scatter` call for `pts` and `vals` on `ax1` (look at how `ax0` and `ax2` are set up for reference).

In [ ]:
fig = plt.figure(figsize=(16, 5))

ax0 = fig.add_subplot(1, 3, 1, projection="3d")
ax0.plot_surface(Xg, Yg, Ztrue, cmap="viridis", linewidth=0, antialiased=True)
ax0.set_title("True function")

ax1 = fig.add_subplot(1, 3, 2, projection="3d")
ax1.plot_surface(Xg, Yg, Zg, cmap="viridis", linewidth=0, antialiased=True, alpha=0.9)
# TODO: scatter the original data points (pts[:, 0], pts[:, 1], vals) in red on ax1
ax1.set_title("RBF interpolant (red = data points)")

ax2 = fig.add_subplot(1, 3, 3, projection="3d")
ax2.plot_surface(Xg, Yg, np.abs(Zg - Ztrue), cmap="inferno", linewidth=0, antialiased=True)
ax2.set_title("Absolute error")

for ax in (ax0, ax1, ax2):
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("f")
    ax.view_init(elev=30, azim=-60)

plt.tight_layout()
plt.show()

## 7. Bonus (open-ended)

Pick at least one and try it:

1. **Implement a second kernel**, the multiquadric RBF $\varphi(r) = \sqrt{1 + (\epsilon r)^2}$, as `multiquadric_rbf(r, epsilon)`. Swap it in for `gaussian_rbf` in section 4 and rerun — how does the surface change?
2. **Sweep `epsilon`.** Rerun sections 4–6 with `epsilon = 0.5`, `2`, `8`, `20`. At what point does `np.linalg.solve` start complaining (or the plot look obviously wrong)? This is the classic RBF ill-conditioning trade-off: small `epsilon` (wide, overlapping bumps) makes $\Phi$ nearly singular.
3. **Fewer points.** Drop `N` to 10. How much worse does the interpolant track the true function?
4. **Noisy data.** Add `vals += rng.normal(0, 0.05, size=N)` before solving. The interpolant will now chase the noise exactly (it still passes through every noisy point!). Look up *ridge regression* / Tikhonov regularization — solving $(\Phi + \lambda I)\mathbf{w} = \mathbf{f}$ instead — and see if a small $\lambda$ gives a smoother, more sensible surface.